<div dir="rtl"><h2>03 — מצב לפני שריקה: פעימה A</h2>
<p>לפני שחוזים משחק, צריך לדעת איך נראתה הליגה בבוקר שלו. <b>M0</b> בונה לכל קבוצה שורה מתוך תוצאות בתאריכים קודמים ממש, לפי שעון ישראל. משחקים מאותו יום אינם רואים זה את זה.</p>
<p><b>טבלה על המגרש, בלי הורדות; עשויה להיות שונה מהטבלה הרשמית.</b> נקודות הן שלוש לניצחון ואחת לתיקו; בשתי עונות החצייה מעגלים למעלה. 11 המשחקים הטכניים אינם קלט לפיצ׳רים.</p>
<p>בפעימה זו: חוקי ליגה, טבלת as_of, רישום הפיצ׳רים ודגל מתמטי שמרני. Elo, כושר, התקפה/הגנה והקשר מהעונה הקודמת יגיעו בפעימות B ו-C. הסטטיסטיקות כאן על development בלבד: <bdi dir="ltr">2006/2007–2023/2024</bdi>. עונות ההחזקה מופיעות רק בשער האימות ההנדסי המאושר.</p>
<ol><li>הגדרות וקלט</li><li>חוקים ואימות</li><li>הטבלה בבוקר המשחק</li><li>רשמי מול ספורטיבי</li><li>יעדים פתוחים ושלב העונה</li><li>שמירה והמשך</li></ol></div>

<div dir="rtl"><h2>סביבת העבודה</h2><p><b>מה התא עושה:</b> טוען את פונקציות החבילה, קובע את שורש הפרויקט ומוודא שימוש ב-venv. הטבלאות מוצגות בעברית וב-RTL.</p></div>

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import HTML, display

from ligat.analysis.prematch_review import (
    championship_examples,
    dead_game_summary,
    postponed_example,
    registry_overview,
    rules_overview,
    split_comparison,
    sporting_official_comparison,
    stage_end_distribution,
    validation_overview,
)
from ligat.analysis.prematch_validation import load_technical_ledger, validate_prematch_tables
from ligat.analysis.prepare import DEVELOPMENT_SEASONS, load_analysis_tables
from ligat.data.repository import Repository
from ligat.features.artifacts import input_fingerprint, save_table_state
from ligat.features.registry import validate_feature_columns
from ligat.features.table_state import TableStateBlock
from ligat.simulate.rules import load_season_rules
from ligat.viz.prematch import dead_games_chart, review_table, stage_end_chart

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Select the project venv kernel"
started = time.perf_counter()
pio.renderers.default = "plotly_mimetype"
display(HTML('<div dir="rtl">סביבת ה-venv אומתה; החישובים מיובאים מהחבילה.</div>'))

<div dir="rtl"><h2>טעינת הקלט לפיצ׳רים</h2><p><b>מה התא עושה:</b> טוען עונות פיתוח דרך load_analysis_tables בלבד. הפיצ׳רים ייבנו ב-DataFrame חדש; הטבלאות הנקיות אינן משתנות. עמודות EDA קיימות, סיכומי סיום ושדות שנדחו אינם נכנסים לחישוב.</p></div>

In [ ]:
tables = load_analysis_tables(seasons=DEVELOPMENT_SEASONS)
rules = load_season_rules()
display(
    HTML(
        f'<div dir="rtl">נטענו {len(tables.match_table):,} משחקים '
        f"ו-{len(tables.team_matches):,} שורות קבוצה×משחק בעונות הפיתוח.</div>"
    )
)

<div dir="rtl"><h2>1. חוקי הליגה וקווי היעד</h2><p><b>מה התא עושה:</b> מציג את הפורמט, החצייה, שוברי השוויון וקווי העליון/הירידה לפי עונה. בפלייאוף מוצגים גודל הקבוצה ומספר המשחקים לכל קבוצה.</p>
<p>ב-<bdi dir="ltr">2011/2012</bdi> ירדו שלוש קבוצות. בשתי העונות שקדמו לה מקום 14 הוביל למבחן ונחשב כאן חלק מאזור הסיכון. מפגש ישיר נשאר שובר שוויון בתקנון; שוויון מוחלט מקבל מיקום ממוצע. בפיצול משתמשים במיקום העונה הקודמת אם עדיין נותר שוויון.</p></div>

In [ ]:
display(review_table(rules_overview(rules, list(DEVELOPMENT_SEASONS))))

<div dir="rtl"><h2>2. שער האימות — טבלאות רשמיות כבדיקה בלבד</h2><p><b>מה התא עושה:</b> משווה 9,956 שורות מחזור בעונות <bdi dir="ltr">2006/2007–2025/2026</bdi>, וכן טבלאות סיום. זו בדיקת הנדסה; אין מחקר או בחירת פיצ׳רים על עונות ההחזקה.</p>
<p>מונים נבדקים במחזורים ללא משחק שחוצה מחזורים. המשחקים הטכניים משוחזרים בספר נפרד לצורך הסבר הפער בלבד. מיקום רשמי נבדק ללא הורדות. בקצות השלבים האימות קפדני; באמצע העונה מותר סדר תצוגה אחר רק בתוך קבוצות שוות בנקודות ובהפרש, עם סף עצירה של 1%.</p>
<p>ארטיפקט מקור מאומת ב-<bdi dir="ltr">2006/2007</bdi>: בטבלה הרשמית חסרים שערי חלק ממשחקי התיקו. מותר רק פער שווה בשערי הזכות והחובה; נקודות, נ/ת/ה והפרש נשארים מדויקים. התוצאות עצמן אומתו מול אירועי השערים. שום כלל דירוג אינו משתנה בגלל אופן הצגת הטבלה באתר.</p><p><b>תוצאות האימות:</b> במחזורים הנקיים 7,572 מתוך 8,196 שורות תואמות ישירות (92.39%). יתר השורות מוסברות ב-467 פערים טכניים וב-157 ארטיפקטים; כולן עברו לפי המדיניות שאושרה. 1,760 שורות של מחזורים חוצים הוחרגו. נקודות הסיום עברו בכל 280 השורות אחרי הסבר הטכניים. סדר תצוגה: 12 מתוך 9,432 שורות ביניים (0.13%), מתחת לסף 1%.</p><p>בדיקה הנדסית בלבד של rules.py מול הטבלאות הרשמיות; אין כאן מודל, ציון או בחירה, ולכן היא אינה חלק מהאיסור על עונות 26–28 במחקר.</p></div>

In [ ]:
engineering_tables = load_analysis_tables(seasons=list(range(8, 28)))
repo = Repository()
official_rounds = repo.read_filtered("round_tables", [("season_id", "in", list(range(8, 28)))])
official_finals = repo.read_filtered("final_tables", [("season_id", "in", list(range(8, 28)))])
validation = validate_prematch_tables(
    engineering_tables,
    rules,
    official_rounds,
    official_finals,
    load_technical_ledger(list(range(8, 28))),
)
assert len(official_rounds) == 9956
display(review_table(validation_overview(validation)))
display(
    HTML(
        '<div dir="rtl"><b>שער האימות עבר.</b> ״זהה בספורטיבי״ מציין התאמה ישירה; '
        "פער טכני וארטיפקט מקור הם פערים מוסברים שנבדקו, "
        "ומחזורים חוצים אינם נכללים באחוז ההתאמה.</div>"
    )
)

<div dir="rtl"><h2>3. בניית הטבלה בבוקר המשחק</h2><p><b>מה התא עושה:</b> יוצר שתי שורות לכל משחק, עם מפתחות game_id ו-team_id. כל שורה משתמשת רק במשחקים מתאריך קודם באותה עונה. הפערים חתומים: נקודות קו היעד פחות נקודות הקבוצה; ערך חיובי משמעו שהקבוצה מפגרת בנקודות אחרי הקו.</p>
<p>בפלייאוף, המיקום הכללי הוא היסט הקבוצה בטבלה ועוד המיקום בתוכה, גם אם לקבוצה בתחתון יותר נקודות מקבוצה בעליון. לכן פער בנקודות בין קבוצות פלייאוף אינו אפשרות לעבור ביניהן. אחרי הפיצול, פער לקו שאינו נגיש הוא NA: פער לירידה בעליון ובאמצעי, ופער למובילה באמצעי ובתחתון; פער לקו העליון קיים רק בסדיר לפני פיצול. נקודות למשחק הן הנקודות המחושבות, כולל חצייה כשכבר חלה, חלקי המשחקים ששוחקו; לפני משחק ראשון הערך הוא אפס.</p></div>

In [ ]:
build_started = time.perf_counter()
features = TableStateBlock(rules).compute(tables)
build_seconds = time.perf_counter() - build_started
validate_feature_columns(features, "table_state")
assert len(features) == 2 * len(tables.match_table)
display(
    HTML(
        f'<div dir="rtl">נבנו {len(features):,} שורות ו-{len(features.columns)} עמודות '
        f"ב-{build_seconds:.2f} שניות. הקלט והטבלאות הרשמיות אינם משתנים.</div>"
    )
)

<div dir="rtl"><h2>משחק נדחה: מספר המחזור אינו ציר זמן</h2><p><b>השאלה:</b> למה לא לקחת את הטבלה הרשמית של המחזור?</p><p><b>מה התא עושה:</b> מוצא משחק אמיתי של קבוצה שמשחק ממחזור קודם שלה הושלם רק אחריו, ומשווה בין שורת הבוקר שלה לטבלה הרשמית.</p>
<p>הטבלה הרשמית היא אחרי המחזור, ולכן היא כוללת גם את תוצאת המשחק הנחזה ומשחקים נוספים מאותו מחזור. הדוגמה מבודדת במפורש גם משחק ממחזור קודם ששוחק בעתיד; זו המחשת דליפה, לא השוואה של שני מדדים לאותה נקודת זמן.</p></div>

In [ ]:
postponed_context, postponed_tables = postponed_example(tables, features, official_rounds)
display(review_table(postponed_context))
display(review_table(postponed_tables))

<div dir="rtl"><h2>4. רשמי מול ספורטיבי — הורדות נקודות</h2><p><b>מה התא עושה:</b> מציג עונות שבהן יש הורדות או פער טכני, ומפריד בין הנקודות הספורטיביות, התוספת הטכנית לצורך האימות והנקודות הרשמיות. לצורך המודלים משתמשים רק בספורטיבי.</p>
<p>ב-<bdi dir="ltr">2011/2012</bdi> הורדות נקודות השפיעו על קו הירידה: לכן ״ירדה בפועל״ היא תווית אבחון של הטבלה הרשמית, ולא יעד אימון ספורטיבי. גם המיקום הספורטיבי כאן משתמש בשיבוץ הפלייאוף הידוע בפועל.</p></div>

In [ ]:
official_comparison = sporting_official_comparison(tables, rules, official_finals)
display(
    review_table(
        official_comparison.loc[
            official_comparison.rank_changed | official_comparison.deduction.ne(0)
        ]
    )
)

<div dir="rtl"><h2>השיבוץ ההיסטורי נשמר — הנקודות נשארות ספורטיביות</h2><p><b>מה התא עושה:</b> מציג את כל ההבדלים בין שיבוץ ספורטיבי מחושב לבין השיבוץ מהלוח בפועל, עם הסיבה המאומתת.</p>
<p>ב-<bdi dir="ltr">2010/2011</bdi> הניצחון הטכני של ב״ש על אשדוד נתן לה שלוש נקודות נוספות והכניס אותה לאמצעי. בלעדיו בית״ר הייתה באמצעי וב״ש בתחתון. בפועל בית״ר שיחקה חמישה משחקי תחתון במאבק ירידה וב״ש שלושה משחקי אמצעי. ב-M0 משתמשים בשיבוץ מהלוח כמידע K רק מרגע הפיצול; ב-M4 הפיצול ייגזר מהמסלול המדומה. final_stage נשאר לאימות בלבד.</p></div>

In [ ]:
display(review_table(split_comparison(tables, validation)))

<div dir="rtl"><h2>5. כמה משחקים נשארים בלי יעד תחרותי פתוח?</h2><p><b>מה התא עושה:</b> סופר בכל עונה ושלב משחקים שבהם לאף קבוצה, לאחת או לשתיהן אין יעד פתוח. המכנה הוא משחקים, ולא שתי שורות הקבוצה של כל משחק.</p>
<p><b>״אין על מה לשחק״ כאן הוא דגל מתמטי שמרני, לא ציון מוטיבציה:</b> הוא בודק אליפות, קו עליון וירידה, ובעליון ובעונות בלי פלייאוף גם את קווי המקומות 2, 3 ו-4 (ה-ת10). שוויון נקודות נשאר פתוח. לפני הפיצול מוכיחים פסילת אליפות דרך פסילה מהעליון, ובטיחות דרך הבטחת שיבוץ שאינו תחתון. אחרי הפיצול משתמשים במעטפת נקודות בתוך הקבוצה. השיטה עשויה לזהות הכרעה מאוחר; היא אינה מניחה שקבוצה בטוחה תפסיק להתאמץ.</p>
<p>באמצעי הישן אין יעד רלוונטי, ולכן כל הקבוצות בו מסומנות בלי יעד. בעליון הירידה כבר מוכרעת; בתחתון האליפות והקו העליון כבר מוכרעים.</p><p><b>למה נוספו קווי המקום (ה-ת10)?</b> גם אחרי שהאליפות הוכרעה, קבוצות בעליון עוד נלחמות על הסגנות ועל אירופה. לכן ״מוכרע״ מחייב שהצד של הקבוצה ביחס לכל קו בין המקומות <bdi dir="ltr">2|3, 3|4, 4|5</bdi> יהיה נעול במעטפת הנקודות. המקום הרביעי נכלל תמיד מטעמי שמרנות, בלי לנסות לקבוע כאן מי תקבל כרטיס אירופי; אירופה אינה יעד חיזוי נפרד.</p><p>טבלת ההשוואה בעליון מחושבת כאן מהעמודות בזיכרון: ההגדרה הישנה היא AND של title_decided, cut_decided ו-relegation_decided, והחדשה דורשת גם places_decided. שיעור שורות קבוצה אינו שיעור משחקים; המכנים מוצגים בנפרד.</p><p><b>מגבלה מתועדת — מבחן הירידה:</b> בעונות <bdi dir="ltr">2009/2010–2010/2011</bdi> מקום 14 הולך למבחן ירידה, ומקומות <bdi dir="ltr">15–16</bdi> יורדים ישירות. הדגל בתחתון רואה את שלושתם כצד אחד של קו הירידה. לכן קבוצה שכבר לא יכולה לסיים מעל מקום 14 מסומנת ״מוכרע״, אף שהיא עוד נלחמת על מבחן הירידה ולא על ירידה ישירה. המגבלה נוגעת רק לתחתון בשתי העונות; היא מתועדת כאן ללא שינוי בקוד הדגל.</p></div>

In [ ]:
dead_summary, match_flags = dead_game_summary(features)
display(review_table(dead_summary, percent=("at_least_one_share", "both_share")))
stage_totals = (
    dead_summary.groupby("stage")[["games", "neither", "one", "both"]]
    .sum()
    .reindex(["regular", "upper", "middle", "lower"])
    .reset_index()
)
display(review_table(stage_totals))

legacy_flag = features[["title_decided", "cut_decided", "relegation_decided"]].all(axis=1)
upper_comparison_rows = []
for label, flags in [("לפני ה-ת10", legacy_flag), ("אחרי ה-ת10", features.nothing_to_play)]:
    upper_flags = flags.loc[features.stage.eq("upper")]
    upper_comparison_rows.append(
        {
            "definition": label,
            "team_games": len(upper_flags),
            "dead_team_rows": int(upper_flags.sum()),
            "dead_team_share": float(upper_flags.mean()),
            "both": int(upper_flags.groupby(level="game_id").sum().eq(2).sum()),
        }
    )
upper_comparison = pd.DataFrame(upper_comparison_rows)
display(HTML('<div dir="rtl"><b>בעליון — הדגל לפני ואחרי ה-ת10</b></div>'))
display(review_table(upper_comparison, percent=("dead_team_share",)))
dead_counts = match_flags.dead_teams.value_counts().reindex([0, 1, 2], fill_value=0)
display(
    HTML(
        f'<div dir="rtl">מתוך {len(match_flags):,} משחקים, '
        f"ב-{int(dead_counts[1] + dead_counts[2]):,} "
        f"({100 * (dead_counts[1] + dead_counts[2]) / len(match_flags):.2f}%) "
        "לפחות קבוצה אחת בלי יעד פתוח לפי ההגדרה החדשה.</div>"
    )
)

<div dir="rtl"><h2>התפלגות המשחקים בלי יעד לאורך העונות</h2><p><b>מה התא עושה:</b> מציג את שיעור המשחקים עם לפחות קבוצה אחת בלי יעד פתוח, בנפרד לכל שלב. בריחוף מופיעים גם המכנים ומספר המשחקים שבהם קבוצה אחת או שתיים בלי יעד. זהו תיאור של הדגל שהוגדר, לא אומדן להשפעת מוטיבציה על תוצאות.</p></div>

In [ ]:
dead_figure = dead_games_chart(dead_summary)
display(dead_figure)

<div dir="rtl"><h2>דוגמאות: אלופה שמגיעה למשחק עם תואר מוכרע</h2><p><b>מה התא עושה:</b> לכל עונה מציג את המשחק הראשון של המובילה שבו הדגל כבר מוכיח שהאליפות שלה, וממיין לפי משחקים שנותרו בשלב.</p>
<p><b>זה אינו תאריך ההכרעה המתמטית המדויק:</b> האליפות יכולה להיות מוכרעת בין משחקי האלופה, והדגל השמרני עשוי לזהות זאת מאוחר. אלו דוגמאות מתועדות לפני משחק, ולא דירוג מוחלט של ״האליפות המוקדמת בהיסטוריה״.</p></div>

In [ ]:
display(review_table(championship_examples(tables, features)))

<div dir="rtl"><h2>6. משחקים עד סוף השלב — אחרי המשחק הנחזה</h2><p><b>מה התא עושה:</b> מציג את התפלגות המונה ומוודא שאינו שלילי. games_remaining_stage כולל את המשחק הנחזה; rounds_to_stage_end מפחית אותו. הספירה מחושבת מהפורמט פחות משחקי העבר, ולא ממספר שורות עתידיות.</p>
<p>משחק טכני שהוחרג אינו תוצאה ספורטיבית שנצפתה. לכן אצל קבוצה כזו המונה עשוי להשאיר משחק אחד נוסף גם אחרי משחקה האחרון בפועל. זה מרחיב את מעטפת האפשרויות ומאחר הכרעות; לא ממלאים תוצאה או מעניקים נקודות למשחק שהוצא.</p></div>

In [ ]:
stage_distribution = stage_end_distribution(features)
stage_figure = stage_end_chart(stage_distribution)
display(stage_figure)

<div dir="rtl"><h2>רישום הפיצ׳רים ומעמדם</h2><p><b>מה התא עושה:</b> מציג את הרשימה המפורשת שמותרת בפלט. עמודה לא רשומה, לרבות שופט, מאמן, אצטדיון או אירוע מתוך המשחק הנחזה, מכשילה את הבדיקה. לדגל nothing_to_play יש מעמד ברירת מחדל עבור M1 והנחת Elo, אך הוא עדיין מועמד לאבלציה ב-M2. עצם הרישום אינו הוכחה לתרומה לחיזוי.</p></div>

In [ ]:
display(review_table(registry_overview()))

<div dir="rtl"><h2>טעינת גרסה 2 מהשכבה הנגזרת</h2><p><b>מה התא עושה:</b> מאמת את ה־checksum של המטא־דאטה מול העותק הקפוא שאושר בפעימה A, ואת קובץ הטבלה מול ה־hash שבמטא־דאטה. טוען את הטבלה ומשווה אותה לחישוב בזיכרון, תוך נרמול טיפוס מספרי המזהים באינדקס בלבד (Int64/int64 לאחר Parquet). הערכים וטיפוסי כל עמודות הנתונים נבדקים בדיוק. אין דריסה של גרסה 2 או של הגרפים הקיימים. רק אם קובץ הטבלה חסר, נוצרת גרסה ונרשמת הודעה מפורשת.</p><p><b>מה רואים:</b> שהעותק השמור מתאים בדיוק למצב המחושב. אין משמעות כדורגלנית לבדיקה הטכנית.</p></div>

In [ ]:
from ligat.analysis.prematch_b_review import assert_same_table_state, load_table_state_snapshot

artifact = Path("data/derived/prematch/table_state_v2.parquet")
metadata = artifact.with_suffix(".metadata.json")
created_table_state = not artifact.exists()
if created_table_state:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    artifact, metadata = save_table_state(
        features, input_sha256=input_fingerprint(tables, rules), commit=commit
    )
    display(HTML('<div dir="rtl"><b>הקובץ היה חסר ולכן נוצר מחדש — יש לדווח.</b></div>'))
else:
    saved_features = load_table_state_snapshot(
        artifact,
        metadata_sha256="d3cfe3f409490b094889953cef461ff3cbdc2fe775fa79ea89cd7f2370049ec0",
    )
    assert_same_table_state(features, saved_features)
    features = saved_features
    display(HTML('<div dir="rtl">גרסה 2 אומתה ונטענה ללא כתיבה; הערכים זהים לחישוב בזיכרון.</div>'))

<div dir="rtl"><h2>מה הושלם ומה ממשיך לפעימה B</h2><p>יש כעת חוזה ברור למצב הטבלה לפני משחק: תוצאות עבר בלבד, שיבוץ היסטורי ידוע, נקודות ספורטיביות, פיצ׳רים רשומים ודגלים עם הוכחה שמרנית. הטבלאות הרשמיות שימשו לבדיקת תקינות בלבד, עם סיווג מפורש של ארטיפקטים, משחקים טכניים, הורדות וסדר תצוגה.</p>
<p>זו תשובה ראשונה לשאלת המחקר 6א ב-VISION: כמה משחקים נשארים בלי אחד היעדים שהוגדרו. כדי לטעון להשפעה על הביצועים, צריך בהמשך קנה מידה של חוזק וציפייה לתוצאה; הדגל לבדו אינו מוכיח ירידת מוטיבציה.</p>
<p>עד כאן פעימה A: מצב טבלה ודגלים, בלי רכיב נלמד. בחלק B1 הבא נבחן את Elo שנבחר ואת ניסוי EXP-001 מתוך התוצרים השמורים, בלי לאמן מחדש. הכושר וההתקפה וההגנה יוסברו בחלק B2 במשימה נפרדת; פעימה C תוסיף הקשר עונתי ושכבת prematch מאוחדת.</p><p><b>ממצא תיאורי:</b> מתוך 4,315 משחקי פיתוח, ב-156 משחקים (3.62%) קבוצה אחת בלי יעד פתוח וב-89 (2.06%) שתיהן; יחד 245 משחקים (5.68%). לדוגמה, מכבי ת״א הגיעה למשחק ב-7.4.2019 עם אליפות מוכחת בדגל, כשנותרו עוד שבעה משחקי שלב אחרי אותו משחק. זהו מועד תצפית לפני משחק שלה, ולא טענה שזה יום ההכרעה בפועל.</p></div>

<div dir="rtl"><h2>B1 — Elo ו־EXP-001: קוראים את הניסוי השמור</h2>
<p><b>מה התא עושה:</b> מייבא כלי תצוגה ומאמת את כל ה־checksums לפני פתיחת הטבלאות והתחזיות.</p><p><b>מה רואים:</b> רק עונות הפיתוח 2006/2007 עד 2023/2024. אין אימון, חיפוש פרמטרים או שחזור Elo.</p>
<p><b>הסבר נתונים:</b> הניסוי כבר הסתיים. בדיקות ה־hash מונעות ערבוב תוצרים מהרצות שונות; גרסת הקוד והקלט נשמרו במטא־דאטה.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לבדיקת שלמות הקבצים.</p></div>

In [ ]:
from ligat.analysis.prematch_b_review import (
    WORDS,
    alignment_summary,
    calibration_bins,
    configurations,
    control_numbers,
    conversion_examples,
    elo_example,
    fold_plan,
    gap_explanations,
    grid_overview,
    load_elo_review,
    prediction_diagnostics,
    score_example,
    score_ruler,
    season_scores,
    selection_comparison,
    top_configuration_contrasts,
)
from ligat.viz.prematch_b import (
    alignment_chart,
    calibration_chart,
    confusion_chart,
    draw_histogram,
    marginal_chart,
    season_score_chart,
    time_weight_chart,
)
from ligat.viz.prematch_b import (
    review_table as b_table,
)

b1_started = time.perf_counter()
elo_review = load_elo_review(ROOT)
b1_figures = {}
display(
    HTML(
        f'<div dir="rtl">כל הקבצים אומתו. {len(elo_review.predictions):,} תחזיות ולידציה של E0050; '
        f"{len(elo_review.prematch):,} שורות Elo לפני משחק בעונות הפיתוח בלבד.</div>"
    )
)

<div dir="rtl"><h2>B1.1 — איך תוצאת משחק מזיזה את Elo</h2>
<p><b>מה התא עושה:</b> מציג חישוב ידני בעזרת הנוסחה הקבועה של E0050.</p><p><b>מה רואים:</b> שתי קבוצות בדירוג 1500; הבית מנצח 2:0. E הוא 0.543 בקירוב והעדכון 6.9 נקודות בקירוב.</p>
<pre dir="ltr">E = 1 / (1 + 10 ** (−(R_home + 30 − R_away) / 400))
S = 1 (home win), 0.5 (draw), 0 (away win)
Δ = 10 × G × 1 × (S − E)
R_home ← R_home + Δ; R_away ← R_away − Δ</pre><p><b>הסבר נתונים:</b> E הוא ניקוד צפוי בסולם 0, חצי, 1 — אינו הסתברות לניצחון. K=10 ממתן את התגובה למשחק יחיד. HFA=30 נכנס לעדכון, לא נוסף שוב בשכבת ההמרה.</p><p><b>הסבר כדורגלני:</b> ניצחון צפוי נותן פחות מניצחון מפתיע. כאן הבית עולה לכ־1506.9 והחוץ יורד לכ־1493.1, והסכום נשמר.</p></div>

In [ ]:
update_example, multiplier_example = elo_example()
display(b_table(update_example))

<div dir="rtl"><h2>B1.1א — הפרש שערים, קיץ ועולות</h2>
<p><b>מה התא עושה:</b> מציג את מכפיל הפרש השערים שבקוד הייצור.</p><p><b>מה רואים:</b> בהפרש 0 או 1 המכפיל 1; בהפרש 2 הוא 1.5; מ־3 ואילך הוא ‎(11+n)/8.</p>
<pre dir="ltr">R_next = 1500 + (1 − 0.33) × (R_end − 1500)</pre><p><b>הסבר נתונים:</b> E0050: נסיגה בקיץ r=0.33; לעולה 1450 קבוע; אין הנחה במשחק מת (d=1). כל משחקי אותו יום משתמשים בדירוגי הבוקר ורק אחר כך מעדכנים.</p><p><b>הסבר כדורגלני:</b> ניצחון בהפרש גדול משפיע יותר. בקיץ דירוג 1600 חוזר ל־1567; קבוצה שחוזרת אחרי ירידה מתחילה כעולה, בלי לשחזר את הדירוג הישן שלה.</p></div>

In [ ]:
display(b_table(multiplier_example))

<div dir="rtl"><h2>B1.2 — מדירוג אחד לשלוש הסתברויות</h2>
<p><b>מה התא עושה:</b> קורא את מקדמי ההמרה הסדורה שנלמדו בכל אחד מ־15 קיפולי האימון.</p><p><b>מה רואים:</b> β מתאר את רגישות ההסתברויות לפער Elo; שני הספים θ₁ ו־θ₂ יוצרים אזור של תיקו.</p>
<pre dir="ltr">x = (R_home − R_away) / 400; σ(z) = 1 / (1 + exp(−z))
P(A) = σ(θ₁ − βx)
P(D) = σ(θ₂ − βx) − σ(θ₁ − βx)
P(H) = 1 − σ(θ₂ − βx)</pre><p><b>הסבר נתונים:</b> בכל קיפול המקדמים נאמדים מעונות האימון בלבד. החוץ הוא הקטגוריה הנמוכה, אחריו תיקו, ואז הבית; הפלט מוצג בסדר בית/תיקו/חוץ.</p><p><b>הסבר כדורגלני:</b> פער Elo גדול יותר מעלה את סיכוי הבית. גם בפער אפס סיכויי הבית והחוץ שונים, כי הביתיות נלמדת בספים.</p></div>

In [ ]:
coefficient_table = elo_review.tables["proposal_converter_effect"].copy()
coefficient_table[WORDS] = (
    "תצורת E0050: פער הדירוג מחולק ב־400; המרה סדורה נלמדת רק מעונות האימון של הקיפול"
)
display(
    b_table(
        coefficient_table,
        columns=["season", "fold", "beta", "theta1", "theta2", "home_probability_change_pp", WORDS],
    )
)

<div dir="rtl"><h2>B1.2א — חיפה 1540 מול עולה 1450: דוגמת הסתברויות</h2>
<p><b>מה התא עושה:</b> מציב פערי דירוג בנוסחה, עם מקדמי קיפול 2023/2024 השמורים.</p><p><b>מה רואים:</b> זו דוגמה היפותטית, לא טענה שכך דורגו הקבוצות ביום מסוים: פער 90 נותן כ־58% / 25% / 17%.</p>
<p><b>הסבר נתונים:</b> ההפרש בין שורת אפס לשורת 100 מודד שינוי בנקודות אחוז, לא תוספת יחסית של אחוזים. בדוגמה האחרונה הוא כ־21.2 נקודות אחוז; על הקיפולים התיאור הוא כ־20.</p><p><b>הסבר כדורגלני:</b> לדירוג החזק יש יתרון, אבל גם לעולה נשאר סיכוי לנצח או לסיים בתיקו.</p></div>

In [ ]:
display(b_table(conversion_examples(elo_review), percent=("בית", "תיקו", "חוץ")))

<div dir="rtl"><h2>B1.3 — השופט: תמיד מאמנים בעבר ובודקים בעונה הבאה</h2>
<p><b>מה התא עושה:</b> קורא את גבולות האימון והוולידציה מהקיפולים השמורים.</p><p><b>מה רואים:</b> 15 קיפולים מתרחבים: תחילה 2006/2007–2008/2009 לאימון, 2009/2010 לוולידציה; לבסוף 2023/2024 לוולידציה.</p>
<p><b>הסבר נתונים:</b> 3721 משחקי ולידציה. הבחירה נעשית לפי log loss משוקלל במספר המשחקים; RPS הוא מדד משני. חמש האחרונות מדווחות גם בנפרד לפי ה־ת6.</p><p><b>הסבר כדורגלני:</b> כך השיטה נשפטת בכל פעם על עונה שעדיין לא נראתה באימון של אותו קיפול.</p></div>

In [ ]:
display(b_table(fold_plan(elo_review)))

<div dir="rtl"><h2>B1.3א — משחק אחד: מה log loss ו־RPS מודדים</h2>
<p><b>מה התא עושה:</b> מחשב ידנית את שני המדדים לתחזית קבועה 58% בית, 25% תיקו, 17% חוץ.</p><p><b>מה רואים:</b> שלוש שורות הן שלוש תוצאות אפשריות לאותו משחק, ולא שלושה משחקים שנצפו.</p>
<pre dir="ltr">LL = −ln(p_observed)
RPS = ((p_H − 1[y=H])² + (p_H+p_D − 1[y∈{H,D}])²) / 2</pre><p><b>הסבר נתונים:</b> log loss הוא מינוס הלוגריתם של ההסתברות לתוצאה שקרתה. RPS הוא ממוצע ריבועי השגיאה בשני הגבולות המצטברים, בסדר בית/תיקו/חוץ. בשניהם נמוך עדיף.</p><p><b>הסבר כדורגלני:</b> תחזית שמותירה סיכוי סביר להפתעה נענשת פחות אם ההפתעה אכן מתרחשת.</p></div>

In [ ]:
display(b_table(score_example()))

<div dir="rtl"><h2>B1.3ב — סרגל איכות: מה אומר log loss של 1.02</h2>
<p><b>מה התא עושה:</b> משווה ייחוס אחיד, B0, E0050 והסף שנקבע מראש.</p><p><b>מה רואים:</b> לכל שורה מוצג גם e בחזקת מינוס log loss: ממוצע גאומטרי של ההסתברות לתוצאה שקרתה.</p>
<p><b>הסבר נתונים:</b> זה ממוצע גאומטרי, ולא ממוצע רגיל. הוא נמוך מהממוצע הרגיל (ב־E0050: 36.0% מול 38.8%), כי כמו log loss הוא מעניש חזק הסתברות נמוכה שניתנה להפתעה. סף 0.98 הוא יעד, לא תוצאת ניסוי.</p><p><b>הסבר כדורגלני:</b> E0050 מבחינה בין משחקים לפי חוזק עבר ומשפרת את הייחוס שאינו יודע מי משחקת.</p></div>

In [ ]:
display(b_table(score_ruler(elo_review), percent=("ממוצע גאומטרי של ההסתברות לתוצאה שקרתה",)))

<div dir="rtl"><h2>B1.4 — מה בדיוק נבדק ב־EXP-001</h2>
<p><b>מה התא עושה:</b> מציג את הרשת שנרצה בפועל, מתוך טבלת הפרמטרים השמורה.</p><p><b>מה רואים:</b> 1280 צירופים בשלב הראשון; 40 בשלב משקלי הזמן, על 10 בסיסי Elo. אחרי איחוד כפילויות: 1310 צירופים שונים.</p>
<p>ניסיון 1 נעצר בבעיית precision loss; ב־B3a-fix אושרה קבלה מותנית רק עם גרדיאנט מנורמל מרבי ≤10⁻⁶, וניסיון 2 רץ מחדש בשלמותו.</p><p><b>הסבר נתונים:</b> כמות הצירופים נובעת מ־4×4×4×2×2×5. שלב משקלי הזמן אינו חיפוש חוזר על כל הרשת. K=10 בקצה הרשת לא הורחב, לפי החלטת עמית; המיטוב מוגבל לרשת שאושרה.</p><p><b>הסבר כדורגלני:</b> בודקים כמה מהר להגיב לתוצאות, מה לשכוח בקיץ ואיך להתחיל עולות; אין נתוני שווי, שחקנים או שופטים.</p></div>

In [ ]:
display(b_table(grid_overview(elo_review)))

<div dir="rtl"><h2>B1.4א — עשר התצורות המובילות</h2>
<p><b>מה התא עושה:</b> קורא את עשר השורות הראשונות של הדירוג המשולב השמור.</p><p><b>מה רואים:</b> הפרשים קטנים מאוד בין הציונים. לכל תצורה מופיע הסבר מילולי קצר, לצד המזהה.</p>
<p><b>הסבר נתונים:</b> E1283 מובילה בממוצע על 15 קיפולים; E0050 שלישית. דירוג מספרי בלבד אינו מספיק לבחירה כשיש תצורות קרובות לפי כלל שגיאת תקן אחת.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית להפרש זעיר בציון בפני עצמו; אין להסיק ממנו שמשקל זמן מסוים הוא חוק של הליגה.</p><p><b>משותף לכל העשר:</b> K=10, נסיגה בקיץ של 33%, מכפיל הפרש שערים ואתחול עולה ב־1450. עמודת ״מה מייחד אותה״ מציגה רק את ההבדלים; עמודת המילים מסבירה את משמעותם. בסיס ההשוואה הוא יתרון בית 30 בעדכון, ללא משקל זמן וללא הנחה במשחק מת. דירוג חמש האחרונות מפורט גם בסעיף ה־ת6.</p></div>

In [ ]:
display(
    b_table(
        top_configuration_contrasts(elo_review),
        columns=[
            "label",
            "rank_all",
            "log_loss",
            "rps",
            "מה מייחד אותה",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>B1.4ב — אפקט שולי ברשת הגסה: תיאור, לא סיבתיות</h2>
<p><b>מה התא עושה:</b> מציג את ממוצע log loss בכל ערך פרמטר, על יתר צירופי הרשת.</p><p><b>מה רואים:</b> שישה פנלים, בסדר ערכי הרשת, עם אותו טווח בציר Y כדי להשוות את גודל הפערים. הריחוף כולל את מספר הצירופים ואת הציון הטוב ביותר באותה קבוצה.</p>
<p><b>הסבר נתונים:</b> זה ממוצע על תצורות, לא אומדן סיבתי. ערך עם הממוצע הנמוך ביותר אינו בהכרח הערך של הצירוף המוביל, כי יש אינטראקציות עם פרמטרים אחרים.</p><p><b>הסבר כדורגלני:</b> למשל, תגובה מהירה מדי למשחקים יכולה להכניס רעש; הגרף מתאר את הרשת שנבדקה ואינו מוכיח מה גורם להצלחה על המגרש.</p><p>בממוצע על כל הצירופים K=20 טוב מעט, אבל הצירוף הטוב ביותר יושב על K=10. הגרף מראה ממוצע, לא את המיטב, ולכן אין סתירה.</p></div>

In [ ]:
b1_figures["marginals"] = marginal_chart(elo_review.tables["stage1_marginals"])
b1_figures["marginals"].show()

<div dir="rtl"><h2>B1.4ג — משקלי זמן על אותם עשרה בסיסי Elo</h2>
<p><b>מה התא עושה:</b> משווה ממוצע ציונים לפי זמן המחצית בשלב השני בלבד.</p><p><b>מה רואים:</b> עשרה צירופי בסיס בכל נקודה: ללא דעיכה או מחצית משקל בכל 1, 2 או 4 עונות.</p>
<p><b>הסבר נתונים:</b> גם כאן הקשר תיאורי ולא סיבתי. שינוי משקל הזמן חל בהתאמת שכבת ההמרה; הוא אינו משנה את סדר עדכוני Elo או מחשב Elo מחדש.</p><p><b>הסבר כדורגלני:</b> כך בודקים אם הקשר בין פער דירוג לתוצאה השתנה לאורך השנים, מעל דירוג Elo שכבר קיים.</p></div>

In [ ]:
b1_figures["time_weights"] = time_weight_chart(elo_review.tables["stage2_table"])
b1_figures["time_weights"].show()

<div dir="rtl"><h2>B1.4ד — הקרובות והבחירה ב־E0050</h2>
<p><b>מה התא עושה:</b> קורא את 13 התצורות שסומנו קרובות ומציג את הפרש הציון ואת שגיאת התקן המזווגת.</p><p><b>מה רואים:</b> הבדיקה מזווגת על אותם משחקים בכל קיפול. קרובה אם הפער הממוצע המשוקלל אינו גדול מ־SE אחד; SE הוא סטיית התקן הלא משוקללת של 15 הפרשי הקיפולים חלקי √15.</p>
<p><b>הסבר נתונים:</b> הבחירה המאושרת היא E0050: בין הקרובות יש לה מספר התוספות המינימלי (אחת: מכפיל הפרש), ובתיקו בפשטות בוחרים log loss נמוך. אין משקל זמן, אתחול עולה דינמי או הנחה במשחק מת.</p><p><b>הסבר כדורגלני:</b> נשמר זיכרון יציב של כוח הקבוצה; ניצחונות בהפרש גדול מקבלים משקל נוסף, בלי להוסיף מנגנונים שלא נדרשו במדגם.</p></div>

In [ ]:
display(
    b_table(
        configurations(elo_review, "close_configurations"),
        columns=["label", "log_loss", "paired_mean_difference", "paired_se", "additions", WORDS],
    )
)

<div dir="rtl"><h2>B1.4ה — E0050 מול B0 בכל עונת ולידציה</h2>
<p><b>מה התא עושה:</b> מציב את ציוני שני הייחוסים על אותו ציר עונות.</p><p><b>מה רואים:</b> B0 משתמש רק בשכיחויות תוצאות האימון. E0050 משתמשת בפער Elo ובהמרה הסדורה; כל נקודה היא עונת ולידציה אחת.</p>
<p><b>הסבר נתונים:</b> הקו מאפשר לראות יציבות ופערים בין עונות; הסיכום הכולל משוקלל במספר המשחקים ואינו ממוצע פשוט של הנקודות בגרף.</p><p><b>הסבר כדורגלני:</b> חוזק הקבוצות מבדיל בין משחק של פייבוריטית למשחק שקול, גם כששכיחות התוצאות הכללית דומה.</p></div>

In [ ]:
b1_figures["season_scores"] = season_score_chart(season_scores(elo_review))
b1_figures["season_scores"].show()

<div dir="rtl"><h2>B1.4ו — ה־ת6: חמש האחרונות אינן בוחרות אותה מובילה</h2>
<p><b>מה התא עושה:</b> מציג את מובילת 15 הקיפולים, מובילת חמש האחרונות ו־E0050.</p><p><b>מה רואים:</b> E1283 ראשונה על כל הקיפולים; E1286 ראשונה בחמש האחרונות. E0050 שלישית בכלל ו־73 בחמש האחרונות.</p>
<p><b>הסבר נתונים:</b> דגל אי־ההסכמה דולק. חמש האחרונות הן דיווח יציבות, בלי סימון קרובות ובלי בחירה מחדש. עמית אישר E0050 לפי כלל הפשטות על 15 הקיפולים.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית ישירה להחלפת המובילה: הטבלה אינה מזהה מה השתנה בליגה.</p></div>

In [ ]:
display(
    b_table(
        selection_comparison(elo_review),
        columns=[
            "label",
            "rank_all",
            "log_loss",
            "rps",
            "rank_last5",
            "last5_log_loss",
            "additions",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>B1.4ז — LOSO: רטרוספקטיבי, לא לבחירה</h2>
<p><b>מה התא עושה:</b> קורא את בדיקת היציבות של עשרת מובילי הרשת הגסה, ללא משקלי זמן.</p><p><b>מה רואים:</b> הכותרת היא אזהרה מהותית: באימון ההמרה נכללות גם עונות מאוחרות מעונת הבדיקה. זו אינה תחזית כרונולוגית.</p>
<p style="font-size:1.4em;color:#9c2c22"><b>⚠️ רטרוספקטיבי, לא לבחירה — ללא משקלי זמן</b></p><p><b>הסבר נתונים:</b> LOSO אינו בוחר או מחליף את walk-forward. הוא מכסה רק את בסיסי Elo ללא משקל זמן; אין להשוות את ציונו כאילו נמדד על אותם קיפולים ומשחקים. דירוג walk-forward כאן הוא ברשת הגסה בלבד: E0050 ראשונה בה, לעומת שלישית בדירוג המשולב עם משקלי הזמן.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לציון LOSO בפני עצמו; הוא אבחון רגישות של ההמרה לעונת האימון.</p></div>

In [ ]:
display(
    b_table(
        configurations(elo_review, "loso_vs_walk_forward"),
        columns=[
            "label",
            "rank_loso",
            "loso_log_loss",
            "rank_walk_forward",
            "walk_forward_log_loss",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>B1.5 — כיול: האם הסתברות חזויה מתאימה לשכיחות בפועל</h2>
<p><b>מה התא עושה:</b> מקבץ את תחזיות E0050 השמורות בעשרה בינים שווי רוחב, בנפרד לבית, לתיקו ולחוץ.</p><p><b>מה רואים:</b> הקו המקווקו הוא כיול מושלם. כל נקודה היא בין לא ריק; בריחוף רואים גם כמה משחקים יש בו.</p>
<p><b>הסבר נתונים:</b> מרחק מהאלכסון מצביע על אי־התאמה מקומית. בינים בקצוות עם מעט משחקים רועשים יותר; הקווים בין הנקודות הם עזר תצוגה ולא התאמה חדשה.</p><p><b>הסבר כדורגלני:</b> כשכתוב 30% לתיקו, רצוי שכשלושה מתוך עשרה משחקים דומים אכן יסתיימו בתיקו; אין צורך שתיקו יהיה התוצאה הסבירה ביותר.</p><p><b>גודל הנקודה מייצג את מספר המשחקים בבין</b>, והמספר המדויק בריחוף. נקודה חלולה מסמנת פחות מ־30 משחקים. נקודה קיצונית בבין קטן, כמו 100% בחוץ או 0% בתיקו, רועשת ואינה מעידה כשלעצמה על חוסר כיול.</p></div>

In [ ]:
b1_figures["calibration"] = calibration_chart(calibration_bins(elo_review.predictions))
b1_figures["calibration"].show()

<div dir="rtl"><h2>B1.5א — עד כ־31% לתיקו: ההסתברות נשארת חיובית</h2>
<p><b>מה התא עושה:</b> מציג את התפלגות הסתברויות התיקו ומחשב סיכום מתוך התחזיות השמורות.</p><p><b>מה רואים:</b> הסתברות התיקו המרבית כ־30.9%; הממוצע כ־27.1%, מול כ־26.8% תיקו שנצפה בוולידציה.</p>
<p><b>הסבר נתונים:</b> ממוצע סביר אינו מוכיח כיול בכל רמת סיכון; לכן מוצג גם גרף הבינים. אין כאן מודל נוסף או התאמה לתוצאות הוולידציה.</p><p><b>הסבר כדורגלני:</b> גם במשחק שקול, סיכוי התיקו נמוך מסיכוי ניצחון של אחת הקבוצות — ועדיין תיקו צפוי להתרחש בהרבה משחקים.</p></div>

In [ ]:
b1_figures["draw_distribution"] = draw_histogram(elo_review.predictions)
b1_figures["draw_distribution"].show()
diagnostic_summary, confusion = prediction_diagnostics(elo_review.predictions)
display(b_table(diagnostic_summary, percent=("accuracy", "max_draw", "observed_draw", "mean_draw")))

<div dir="rtl"><h2>B1.5ב — 47.8% דיוק, ואפס תחזיות תיקו ב־argmax</h2>
<p><b>מה התא עושה:</b> מציג מטריצת בלבול בין התוצאה בפועל לקטגוריה בעלת ההסתברות המרבית.</p><p><b>מה רואים:</b> השורות הן בפועל והעמודות הן argmax. עמודת התיקו כולה אפס; אין פירוש הדבר שהשיטה נותנת הסתברות אפס לתיקו.</p>
<p><b>הסבר נתונים:</b> log loss שופט את כל ההתפלגות ולא רק את הקטגוריה המובילה. היעדר תיקו ב־argmax כשלעצמו אינו פגם במדד; הסתברות נמוכה מדי לתיקו כן תיענש. דיוק אינו השופט לבחירה.</p><p><b>הסבר כדורגלני:</b> בסימולציה מגרילים מכל ההסתברויות, ולכן יתקבלו גם תיקו והפתעות. לא בוחרים אוטומטית את התוצאה בעלת ההסתברות הגבוהה ביותר בכל משחק.</p></div>

In [ ]:
b1_figures["confusion"] = confusion_chart(confusion)
b1_figures["confusion"].show()

<div dir="rtl"><h2>B1.6 — כל הליגה: הדירוג בסיום מול הטבלה בסיום</h2>
<p><b>מה התא עושה:</b> מציג Spearman ומדדי התאמה לכל עונת פיתוח, ואת הסיכום על כל 252 שורות קבוצה־עונה.</p><p><b>מה רואים:</b> דירוג Elo אחרי המשחק האחרון מול מיקום ספורטיבי סופי: 38.5% זהים, 74.2% עד מקום אחד, 89.7% עד שניים.</p>
<p><b>הסבר נתונים:</b> זו בדיקה בדיעבד באותה נקודת סיום, לא חיזוי מקום סופי בתחילת העונה ולא ציון מחוץ למדגם. המיקום ספורטיבי, בלי הורדות נקודות; Spearman מחושב בנפרד בכל עונה.</p><p><b>הסבר כדורגלני:</b> Elo מזהה בדרך כלל את סדרי הכוח אך אינו מחקה במדויק את צבירת הנקודות בעונה. ביתיות, חוזק היריבה וזיכרון העבר משפיעים עליו.</p></div>

In [ ]:
display(
    b_table(
        elo_review.tables["proposal_sporting_alignment"],
        columns=[
            "season",
            "teams",
            "spearman",
            "exact_percent",
            "within_one_percent",
            "within_two_percent",
        ],
    )
)
display(
    b_table(
        alignment_summary(elo_review.tables["proposal_1_all_teams"]),
        percent=("מקום זהה", "עד מקום אחד", "עד שני מקומות"),
    )
)

<div dir="rtl"><h2>B1.6א — פיזור מיקומים: האלכסון הוא התאמה מלאה</h2>
<p><b>מה התא עושה:</b> מציג את כל 252 הקבוצות־עונות על צירי מיקום ספורטיבי ומיקום לפי Elo.</p><p><b>מה רואים:</b> נקודות על האלכסון הן אותו מיקום. ייתכן חפיפה בין עונות; בריחוף יש קבוצה, עונה, דירוג Elo ופער מיקומים.</p>
<p><b>הסבר נתונים:</b> הדירוג נעשה בתוך כל עונה. העונות עם 12, 14 ו־16 קבוצות נשמרות; אין לערבב דירוגים מוחלטים בין עונות.</p><p><b>הסבר כדורגלני:</b> סטייה מהאלכסון מזכירה שכוח לפי תוצאות עבר וטבלת נקודות עונתית הם שני תיאורים שונים של אותה קבוצה.</p></div>

In [ ]:
b1_figures["league_alignment"] = alignment_chart(elo_review.tables["proposal_1_all_teams"])
b1_figures["league_alignment"].show()

<div dir="rtl"><h2>B1.6ב — פער של ארבעה מקומות ומעלה: מה Elo זוכר</h2>
<p><b>מה התא עושה:</b> מציג את כל 14 המקרים שנשמרו, עם פירוש כיוון הפער.</p><p><b>מה רואים:</b> פער חיובי פירושו שהקבוצה סיימה במקום ספורטיבי טוב מהדירוג שנתן לה Elo; פער שלילי הוא הכיוון ההפוך.</p>
<p><b>הסבר נתונים:</b> זה פירוש תיאורי של מנגנון הזיכרון, לא זיהוי סיבתי של מקור ההפתעה. אין כאן מידע על סגל, מאמן או תקציב.</p><p><b>הסבר כדורגלני:</b> Elo סקפטי כלפי הפתעות ונאמן למוניטין שנוצר מתוצאות עבר, לא לשם המועדון. למשל חדרה ב־2018/2019 מסיימת שישית אך מדורגת 12 לפי Elo, אחרי פתיחה כעולה ב־1450; בני יהודה ב־2013/2014 מסיימת 14 אך Elo עדיין מדרג אותה שישית.</p></div>

In [ ]:
display(
    b_table(
        gap_explanations(elo_review),
        columns=[
            "season",
            "team_name",
            "elo_start",
            "elo_end",
            "position",
            "elo_rank",
            "position_difference",
            "הסבר כדורגלני",
        ],
    )
)

<div dir="rtl"><h2>B1 — מספרי בקרה ושמירת הגרפים</h2>
<p><b>מה התא עושה:</b> מציג מספרי בקרה מדויקים ושומר רק את גרפי B1 החדשים ל־HTML ול־PNG.</p><p><b>מה רואים:</b> E0050: ‏1.020424 ו־0.208413; B0: ‏1.084974. שלוש התאמות המקום מוצגות באחוזים.</p>
<p><b>הסבר נתונים:</b> המספרים מחושבים מתוצרי הניסוי המאומתים. בשלב זה לא נבנה חלק B2, ולא נבחר או אומן רכיב חדש.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לפעולת השמירה. ההמשך המאושר בנפרד יסביר את הכושר ואת ההתקפה וההגנה מעל הבסיס של Elo.</p></div>

In [ ]:
display(b_table(control_numbers(elo_review)))
b1_figure_dir = Path("reports/figures/prematch_b1")
b1_figure_dir.mkdir(parents=True, exist_ok=True)
for name, figure in b1_figures.items():
    figure.write_html(b1_figure_dir / f"{name}.html", include_plotlyjs=True)
    figure.write_image(
        b1_figure_dir / f"{name}.png", width=1700, height=figure.layout.height or 600, scale=1.5
    )
display(
    HTML(
        f'<div dir="rtl">נשמרו {len(b1_figures)} גרפים. זמן חלק B1: '
        f"{time.perf_counter() - b1_started:.2f} שניות.</div>"
    )
)

<div dir="rtl"><h2>B2 — כושר, התקפה והגנה: קוראים את EXP-002</h2><p><b>מה התא עושה:</b> מאמת את ה־checksums של הניסוי ושל form_v1, וטוען רק פיצ׳רים, מקדמים ותחזיות שמורים.</p><p><b>מה רואים:</b> אותן 18 עונות פיתוח: 2006/2007 עד 2023/2024. אין טעינה מחודשת של נתוני משחקים נקיים בחלק B2, ואין אימון.</p><p><b>הסבר נתונים:</b> כל הטבלאות נגזרות מקבצים שנשמרו ב־B4a וב־B4c. נתיב תחזיות EXP-002 מרוכז בקבוע אחד במודול, לקראת המעבר ב־B6. חריגת האימות ההנדסי של A נשארת כפי שאושרה.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לבדיקת ה־hash; היא מוודאת שההסבר נשען על אותו ניסוי שאושר.</p></div>

In [ ]:
from ligat.analysis.prematch_form_review import (
    SELECTED_P,
    candidate_table,
    coefficient_summary,
    decision_table,
    displayed_controls,
    draw_diagnostics,
    ewma_illustrations,
    haifa_season,
    load_form_review,
    opening_coefficients,
    points_coefficients,
    points_leaders,
    poisson_coefficients,
    reported_summer_check,
    route_overview,
    route_references,
)
from ligat.viz.prematch_form import (
    addition_chart,
    form_table,
    haifa_chart,
    recent_scatter,
    rho_chart,
)

b2_started = time.perf_counter()
form_review = load_form_review(ROOT)
b2_figures = {}
display(
    HTML(
        f'<div dir="rtl">הקבצים אומתו: {len(form_review.form):,} שורות × '
        f"{len(form_review.form.columns)} פיצ׳רים; {len(form_review.predictions):,} "
        "תחזיות ולידציה לתצורה שנבחרה.</div>"
    )
)

<div dir="rtl"><h2>B2.1 — ערכי פתיחה: כמה מהכוח נשמר בין עונות</h2><p><b>מה התא עושה:</b> מציג את תשעת ערכי ρ לכל עונה מתוך המטא־דאטה של form_v1.</p><p><b>מה רואים:</b> שלושה מדדים — הבקעה, ספיגה ונקודות — וכל אחד לכל המשחקים, לבית ולחוץ בנפרד. לקראת 2023/2024 ערכי all הם 0.677, 0.510 ו־0.762.</p><pre dir="ltr">prior = L_previous + ρ × (team_previous − L_previous)</pre><p><b>הסבר נתונים:</b> ρ נלמד רק מזוגות עונות קודמים לקראת העונה הנחזית, ב־OLS דרך הראשית על פערים מממוצע הליגה, עם חסימה ל־<bdi dir="ltr">[0,1]</bdi>. סימון חלול הוא גיבוי 0.5 בתחילת הדאטה; בריחוף רואים את גודל המדגם. אלו אינם המקדמים של בדיקת הקיץ שתופיע בהמשך.</p><p><b>הסבר כדורגלני:</b> קבוצה שהבקיעה מעל הממוצע מתחילה עם חלק מהיתרון שלה, ולא חוזרת מיד לקבוצה ממוצעת. לדוגמה, עם ρ התקפה 0.677, פער עבר של חצי שער מצטמצם לכ־0.34 שער בפתיחה.</p></div>

In [ ]:
rho_values = opening_coefficients(form_review)
b2_figures["rho"] = rho_chart(rho_values)
b2_figures["rho"].show()

<div dir="rtl"><h2>B2.1א — עולות: מתחילות ממידע על עולות קודמות</h2><p><b>מה התא עושה:</b> מציג את ערכי הליגה, ρ וערכי העולה שהיו זמינים לקראת 2023/2024.</p><p><b>מה רואים:</b> כל תשעת המדדים והגרסאות. ערך העולה נלמד מעונות שכבר הסתיימו; אין שימוש בעונה שבה היא עולה.</p><p><b>הסבר נתונים:</b> ממוצע הליגה בטבלה הוא של 2022/2023; ממוצע העולות מאגד את העונות הראשונות של עולות קודמות. לכן אלה אינם שני מדגמים מאותה תקופה. כשיש פחות מארבע עולות קודמות, משתמשים בשלוש האחרונות בטבלה הספורטיבית של העונה הקודמת. בעונת הדאטה הראשונה הפתיחה ניטרלית, כמוגדר ב־T-010.</p><p><b>הסבר כדורגלני:</b> בבדיקה התיאורית הכללית של Claude שצוטטה ב־T-010, עולה הבקיעה כ־1.03 מול 1.22 בליגה, ספגה כ־1.45 מול 1.23 וצברה כ־1.09 נקודות מול 1.36. כלומר כ־0.2 ספיגות נוספות למשחק. זה תיאור כללי של העולות; הוא אינו ההפרש בין שתי עמודות הפתיחה של 2023/2024 בטבלה.</p></div>

In [ ]:
display(
    form_table(
        rho_values.loc[rho_values.season_id.eq(25)],
        columns=[
            "season",
            "metric",
            "stratum",
            "rho",
            "league_mean",
            "promoted_value",
            "promoted_observations",
            "promoted_fallback",
        ],
    )
)

<div dir="rtl"><h2>B2.1ב — הקיץ: הכוח שורד, המומנטום כמעט לא מוסיף</h2><p><b>מה התא עושה:</b> מציג את הבדיקה המדווחת של Claude, עם הגדרת המשתנים שאישר עמית; אינו מתאים רגרסיה מחדש.</p><p><b>מה רואים:</b> 203 קבוצות־עונה שנשארו בליגה, בעונות <bdi dir="ltr">2007/2008–2023/2024</bdi>. המשתנה הנחזה: נקודות ספורטיביות למשחק בעשרת המשחקים הראשונים של העונה.</p><p><b>הסבר נתונים:</b> אותה רגרסיה ליניארית עם חותך וללא משקלים משתמשת בשני מנבאים: נקודות למשחק בכל העונה הקודמת ובעשרת האחרונים שלה, לפי תאריך וכולל פלייאוף. עשרת האחרונים כלולים גם בעונה המלאה: 0.11 מודד את התוספת שלהם אחרי שהעונה כולה כבר נלקחה בחשבון. המתאמים הפשוטים 0.54 ו־0.37 הם מדד אחר. המקור הוא T-010 §8.4 והבהרת עמית במשימה זו; אין כאן ציון חיזוי חדש.</p><p><b>הסבר כדורגלני:</b> קצב הנקודות לאורך עונה מלאה מספר על חוזק הקבוצה יותר מסיום העונה בלבד. לכן החלונות אינם חוצים את הקיץ: מתחילים מערך פתיחה שמתאר כוח, ואז לומדים מחדש את העונה הנוכחית.</p></div>

In [ ]:
display(form_table(reported_summer_check()))

<div dir="rtl"><h2>B2.2 — EWMA: זיכרון שדועך בהדרגה</h2><p><b>מה התא עושה:</b> מחשב משקלים מתמטיים בלבד לשלושה זמני מחצית ולחלון 5.</p><p><b>מה רואים:</b> משחק 1 הוא המשחק האחרון שהסתיים לפני יום המשחק; משחק 5 הוא החמישי לאחור. השורה האחרונה מרכזת את המשקל של חמשת האחרונים יחד.</p><pre dir="ltr">α = 1 − 0.5 ** (1/h)
value_new = α × result + (1−α) × value_old
weight(age=j) = α × (1−α) ** (j−1)</pre><p><b>הסבר נתונים:</b> EWMA הוא ממוצע מתעדכן: כל תוצאה חדשה מקבלת α, והעבר מוכפל ב־1−α. זמן מחצית 5 אינו חלון 5: חמשת האחרונים נותנים בו רק 50% מהמשקל, לעומת 100% בחלון מלא של 5. בגרסת venue סופרים משחקים באותו מגרש בלבד.</p><p><b>הסבר כדורגלני:</b> ניצחון או הפסד משפיעים מיד, ואז חשיבותם דועכת בלי להיעלם בבת אחת. בגרסת בית, משחק חוץ אינו משנה את הממוצע של משחקי הבית.</p></div>

In [ ]:
weight_table, prior_weight_table = ewma_illustrations()
display(form_table(weight_table, percent=("EWMA 5", "EWMA 10", "EWMA 20", "חלון 5")))

<div dir="rtl"><h2>B2.2א — כמה נשאר מהדעה של תחילת העונה</h2><p><b>מה התא עושה:</b> מחשב את חלקו של ערך הפתיחה אחרי 5, 10, 20 ו־30 משחקים.</p><p><b>מה רואים:</b> ב־EWMA 5 נשאר חצי מהפתיחה אחרי חמישה משחקים ורבע אחרי עשרה. בחלון 5 הפתיחה נעלמת לחלוטין אחרי חמישה.</p><pre dir="ltr">prior_mass(k) = (1−α) ** k = 0.5 ** (k/h)</pre><p><b>הסבר נתונים:</b> המשקל שנשאר לפתיחה אחרי k תוצאות הוא ‎(1−α)^k. המונה הוא מספר המשחקים באותה גרסה, לא מחזור הליגה; כל המועמדים מאופסים בקיץ לפי החוזה.</p><p><b>הסבר כדורגלני:</b> בתחילת העונה עדיין יודעים מעט על הקבוצה החדשה. כשמצטברים משחקים, התוצאות העדכניות מחליפות בהדרגה את ההערכה שהגיעה מהעונה הקודמת.</p></div>

In [ ]:
display(form_table(prior_weight_table, percent=tuple(f"אחרי {k} משחקים" for k in (5, 10, 20, 30))))

<div dir="rtl"><h2>B2.3 — מכבי חיפה 2023/2024: ההתקפה לפני כל משחק</h2><p><b>מה התא עושה:</b> מחבר את שורות form_v1 לדירוגי הזמן ולשערים בפועל שנשמרו בניסוי.</p><p><b>מה רואים:</b> EWMA 5 בבית ובחוץ כשני קווים, EWMA 20 על כל המשחקים כקו מקווקו, והתוצאות בפועל כסימני ×.</p><p><b>הסבר נתונים:</b> אלו פיצ׳רים של קצב הבקעה, לא λ של מודל השערים. קו בית מופיע רק במועדי משחקי הבית וקו חוץ רק בחוץ; אין השלמה או חישוב מחדש של ערכי venue בתאריכים אחרים. סימני השערים בפועל מוצגים להשוואה אחרי המשחק ואינם קלט לפיצ׳ר של אותו משחק.</p><p><b>הסבר כדורגלני:</b> ההפרדה מאפשרת לתאר קבוצה אחרת בבית ובחוץ. EWMA 20/all מגיב לאט יותר ומחליק את ההבדלים, כי הוא זוכר יותר משחקים ומערבב מגרשים.</p></div>

In [ ]:
haifa_example = haifa_season(form_review)
b2_figures["haifa_attack"] = haifa_chart(haifa_example, "gf")
b2_figures["haifa_attack"].show()

<div dir="rtl"><h2>B2.3א — מכבי חיפה: ההגנה בבית הגיעה לכ־0.41</h2><p><b>מה התא עושה:</b> מציג את אותה השוואה עבור שערי חובה.</p><p><b>מה רואים:</b> הספיגות בפועל הן נקודות, והקווים הם ההערכות שהיו זמינות לפני כל משחק. נמוך יותר הוא הגנה שסופגת פחות.</p><p><b>הסבר נתונים:</b> הערך הנמוך בבית הוא כ־0.406 לפני משחק, ולכן מעוגל ל־0.41. שינויי הקו נובעים מהמשחקים שכבר הסתיימו בלבד; זה אינו אומדן של השפעת אצטדיון או של סיבה אחרת.</p><p><b>הסבר כדורגלני:</b> במונחי הפיצ׳ר, חיפה מציגה ״מבצר בבית״: ממוצע ספיגות נמוך מאוד בגרסת הבית, שאינו נבלע בממוצע הכללי והאיטי.</p></div>

In [ ]:
b2_figures["haifa_defense"] = haifa_chart(haifa_example, "ga")
b2_figures["haifa_defense"].show()
display(
    HTML(
        f'<div dir="rtl">המינימום בפיצ׳ר הגנת הבית לפני משחק: '
        f"{haifa_example.loc[haifa_example.is_home, 'ga_venue_ewma5'].min():.6f} שערים למשחק.</div>"
    )
)

<div dir="rtl"><h2>B2.4 — שני מסלולים, שתי שאלות ושני סרגלים</h2><p><b>מה התא עושה:</b> מציג את מבנה EXP-002 ואת ציוני הייחוס של כל מסלול.</p><p><b>מה רואים:</b> P בודק התקפה והגנה לפי כמות השערים; O בודק אם קצב הנקודות מוסיף להסתברויות התוצאה מעבר ל־Elo.</p><p><b>הסבר נתונים:</b> ב־P מדד השיפוט הוא Poisson NLL לשורת קבוצה־משחק, כולל האיבר log(y!). ב־O זה log loss למשחק על בית/תיקו/חוץ. הערכים אינם באותן יחידות ואין להשוות ביניהם. שני המסלולים משתמשים באותם 15 קיפולי פיתוח; כאן רק קוראים את הציונים שנשמרו.</p><p><b>הסבר כדורגלני:</b> אפשר לשפר את ההערכה כמה שערים יובקעו בלי לשפר באותה מידה את תוצאת המשחק; כמות שערים ותוצאת בית/תיקו/חוץ הן שאלות שונות.</p></div>

In [ ]:
display(form_table(route_overview()))
form_references = route_references(form_review)
display(form_table(form_references))

<div dir="rtl"><h2>B2.4א — P: כל 22 המועמדים עם Elo</h2><p><b>מה התא עושה:</b> קורא את הדירוג השמור ומחשב הפרשים מזווגים מהציונים השמורים בכל קיפול.</p><p><b>מה רואים:</b> לכל מועמד: נוסחה במילים, ההבדל מהאחרים, NLL, תוספת מעל Elo ביחידות SE, מספר עונות עם שיפור ודירוג חמש האחרונות. קרבה נשמרת רק מול מוביל 15 הקיפולים.</p><pre dir="ltr">d_f = NLL(Elo reference, f) − NLL(candidate + Elo, f)
z = weighted_mean(d_f) / (std(d_f, ddof=1) / √15)</pre><p><b>הסבר נתונים:</b> לכל עונה d הוא NLL של ייחוס Elo פחות NLL של המועמד. הממוצע משוקלל במספר שורות הקבוצה; SE היא סטיית התקן הלא משוקללת של 15 ההפרשים חלקי √15. z הוא היחס ביניהם, לא p-value. המספרים משקפים פיתוח שבו נבדקו 22 אפשרויות, לא אישור על עונת מבחן חדשה.</p><p><b>הסבר כדורגלני:</b> החלונות בודקים אם כמה שהקבוצה מבקיעה וכמה שהיריבה סופגת נותנים חידוד מעל דירוג הכוח. בשני צדי ההשוואה יש Elo.</p></div>

In [ ]:
p_candidates = candidate_table(form_review, "P")
display(
    form_table(
        p_candidates,
        columns=[
            "method",
            "מה מייחד אותה",
            "poisson_nll",
            "z",
            "seasons_better",
            "last5_poisson_nll",
            "rank_last5",
            "within_one_se",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>B2.4ב — רק מידע מתוך העונה מוסיף מעל Elo</h2><p><b>מה התא עושה:</b> מציג את יחס השיפור לשגיאת התקן לכל 22 המועמדים, בצבע לפי מקור המידע.</p><p><b>מה רואים:</b> כל 18 המועמדים שמשתמשים בתוצאות העונה נמצאים מעל שגיאת תקן אחת. ארבעת prev/hist אינם מוסיפים בממוצע מעל Elo.</p><p><b>הסבר נתונים:</b> הציר חיובי כשהמועמד מקטין NLL מעבר לייחוס Elo. זהו תיאור של ההפרשים בקיפולי הפיתוח, ולא בדיקת סיבתיות או z-test עם p-value. כלל הקרבה למוביל הוא מבחן אחר, שמופיע בטבלה.</p><p><b>הסבר כדורגלני:</b> Elo כבר זוכר היסטוריה ארוכה של תוצאות. תיאור ההבקעה והספיגה בעונה הנוכחית נותן מידע נוסף; חזרה על ממוצעי העונות הישנות אינה מוסיפה כאן.</p></div>

In [ ]:
b2_figures["addition_z"] = addition_chart(p_candidates)
b2_figures["addition_z"].show()

<div dir="rtl"><h2>B2.4ג — ה־ת6: 15 העונות מול חמש האחרונות</h2><p><b>מה התא עושה:</b> מציב את NLL של כל מועמד על שני צירים, עם ייחוס Elo בשניהם.</p><p><b>מה רואים:</b> ewma5/all מוביל לפי הכלל על 15 הקיפולים ומדורג 21 בחמש האחרונות; ewma5/venue מוביל בחמש האחרונות, ונבחר בידי עמית.</p><p><b>הסבר נתונים:</b> נמוך עדיף בשני הצירים. דגל ה־ת6 דולק משום שהדירוגים אינם מסכימים. בחמש האחרונות ההבדלים בין המועמדים קטנים מהרעש; אין כאן סימון קרבה חדש או הכרעה סטטיסטית לפי חמש עונות בלבד.</p><p><b>הסבר כדורגלני:</b> חוזק כללי והפרדה בין בית לחוץ נותנים תמונה שונה בתקופות שונות. הגרף מציג את אי־היציבות, ואינו מסביר לבדו מה השתנה בליגה.</p></div>

In [ ]:
b2_figures["recent_scatter"] = recent_scatter(p_candidates, form_references)
b2_figures["recent_scatter"].show()

<div dir="rtl"><h2>B2.4ד — הבחירה המאושרת: ewma5/venue</h2><p><b>מה התא עושה:</b> מעמיד את המוביל לפי כלל הפשטות מול התצורה שנבחרה, בלי לבחור מחדש.</p><p><b>מה רואים:</b> ewma5/venue: ‏NLL 1.409455 ותוספת של כ־2.95 שגיאות תקן מעל Elo. הוא אינו בין הקרובים למוביל על 15 הקיפולים.</p><p><b>הסבר נתונים:</b> זו חריגה מודעת מכלל הפשטות, שהכריע עמית אחרי QA: עדיפות לחיזוי העתיד ולעקביות התוספת. התצורה נבחרה למרות ש־ewma5/all מוביל בממוצע הארוך. הפערים הקטנים בחמש האחרונות נשארים הסתייגות מפורשת.</p><p><b>הסבר כדורגלני:</b> הבחירה שומרת קצב הבקעה וספיגה נפרד לבית ולחוץ, עם זיכרון קצר יחסית, כדי לייצג הבדלים עדכניים בין שתי סביבות המשחק.</p></div>

In [ ]:
display(
    form_table(
        p_candidates.loc[p_candidates.config_id.isin(["P_ewma5_all_elo", SELECTED_P])],
        columns=[
            "method",
            "rank_all",
            "poisson_nll",
            "z",
            "seasons_better",
            "rank_last5",
            "last5_poisson_nll",
            "within_one_se",
            WORDS,
        ],
    )
)

<div dir="rtl"><h2>B2.4ה — מקדמי הנבחר בכל קיפול</h2><p><b>מה התא עושה:</b> קורא את a, d, b ו־γ מתוך fits.parquet של ewma5/venue.</p><p><b>מה רואים:</b> a מתייחס להתקפת הקבוצה, d להגנת היריבה, b לפער Elo ו־γ לביתיות. כל שורה נלמדה רק מעונות האימון של אותו קיפול.</p><pre dir="ltr">log λ_X = c + γ × home + a × log(gf_X+0.1)
          + d × log(ga_opponent+0.1) + b × Elo_difference_X/400</pre><p><b>הסבר נתונים:</b> בכל מועמדי P הוגדר מראש log(x+0.1), כדי לאפשר גם חלון עם אפס שערים. ההיסט אינו משנה את form_v1 ואינו פרמטר שנבחר בניסוי. המקדמים מתארים קשרים מותנים במודל השיפוט; הם אינם משקלים שמחליפים את הפיצ׳ר ב־M0.</p><p><b>הסבר כדורגלני:</b> התקפה טובה והגנה שסופגת הרבה אצל היריבה שתיהן מעלות את ציפיית השערים; Elo מוסיף את הפרש הכוחות מעבר להן.</p></div>

In [ ]:
selected_coefficients = poisson_coefficients(form_review)
display(form_table(selected_coefficients, columns=["season", "a", "d", "b", "gamma", WORDS]))

<div dir="rtl"><h2>B2.4ו — תרגום המקדמים לשערים ולהסתברות ניצחון</h2><p><b>מה התא עושה:</b> מציג חציון וטווח על 15 הקיפולים, עם נקודת הייחוס שנקבעה מראש.</p><p><b>מה רואים:</b> עלייה של gf מ־1.5 ל־1.65: בחציון כ־1.39% תוספת שערים וכ־0.50 נקודות אחוז לניצחון בית. עלייה מקבילה בספיגות היריבה נותנת כ־0.87% תוספת שערים.</p><pre dir="ltr">λ_after / λ_before = ((1.65+0.1)/(1.5+0.1)) ** a
ΔP(home win) = 100 × (P_after − P_before)  [percentage points]</pre><p><b>הסבר נתונים:</b> בדוגמה Elo=0; יתר משתני ההבקעה והספיגה בממוצע הליגה של עונות האימון של כל קיפול. ממוצע זה משוחזר מממוצעי העונות השמורים ומספר שורות Elo, בלי קריאת תוצאות נקיות. ההסתברויות מחושבות ברשת <bdi dir="ltr">0–10</bdi> המנורמלת הקיימת. אלה המחשות מותנות, לא השפעות סיבתיות.</p><p><b>הסבר כדורגלני:</b> בגרסת venue כל ממוצע נשען על חצי מהמשחקים ולכן רועש יותר; המודל נותן לו מקדם קטן יותר ומשאיר יותר משקל ל־Elo: b בחציון כ־1.15 מול כ־1.0 ב־all. למשל, שיפור של 10% בקצב ההבקעה אינו הופך במודל מיד לשיפור של 10% בשערים הצפויים.</p></div>

In [ ]:
display(form_table(coefficient_summary(selected_coefficients)))

<div dir="rtl"><h2>B2.5 — כושר בנקודות: 0 מתוך 22 מוסיפים בממוצע</h2><p><b>מה התא עושה:</b> מציג את ייחוס Elo, שלושת המובילים במסלול O ואת הצעת הפשטות.</p><p><b>מה רואים:</b> גם המוביל בין מועמדי הכושר גרוע מעט מ־E0050 בממוצע 15 הקיפולים. EWMA 20/all הוא הצעת הפשטות של O, ולא המוביל ולא פיצ׳ר שנבחר.</p><p><b>הסבר נתונים:</b> זו תוצאה על תוספת ליניארית של הפרש נקודות למשחק להמרה הסדורה של פער Elo. מתאם עם התוצאה לבדו אינו מספיק: כאן בודקים מה נשאר אחרי ש־Elo כבר במודל.</p><p><b>הסבר כדורגלני:</b> צבירת נקודות חוזרת במידה רבה על המידע מתוצאות ש־Elo כבר מסכם. בניסוי הזה היא לא סיפקה תוספת ממוצעת מעליו.</p></div>

In [ ]:
o_candidates = candidate_table(form_review, "O")
display(form_table(points_leaders(form_review)))
display(
    HTML(
        f'<div dir="rtl">מועמדים עם שיפור ממוצע מעל Elo: '
        f"{int(o_candidates.improvement.gt(0).sum())} מתוך {len(o_candidates)}.</div>"
    )
)

<div dir="rtl"><h2>B2.5א — מדוע בדוגמת הכושר מתקבל מקדם שלילי</h2><p><b>מה התא עושה:</b> קורא את שני המקדמים של הצעת הפשטות O: ‏ewma20/all, לכל קיפול.</p><p><b>מה רואים:</b> בקיפול הראשון β של Elo מגיע לכ־6.1, ולכושר מקדם שלילי. הטבלה מתייחסת לדוגמה זו במפורש, ולא לכל 22 המועמדים.</p><p><b>הסבר נתונים:</b> קולינאריות פירושה ששני משתנים חולקים מידע דומה. Elo וקצב הנקודות מסכמים שניהם תוצאות עבר, ולכן כל מקדם מתאר את המשתנה לאחר שליטה באחר. מקדמים גדולים או שליליים אינם פירוש עצמאי של איכות קבוצה.</p><p><b>הסבר כדורגלני:</b> השערה אפשרית: נקודות ש־Elo אינו מאשר הן מזל שחוזר לממוצע. <b>זו השערה שלא נבדקה ישירות</b>, ולא סיבה שהוכחה בטבלה. ההחלטה המאושרת: כושר בנקודות אינו נכנס ל־M0; נשאר 🔬 רק לבנצ׳מרק XGBoost וייצא אם לא יתרום.</p></div>

In [ ]:
display(form_table(points_coefficients(form_review)))

<div dir="rtl"><h2>B2.6 — התיקו ב־Poisson: חזוי מול בפועל</h2><p><b>מה התא עושה:</b> מחשב שיעורי בית/תיקו/חוץ מהתחזיות השמורות של P_ewma5_venue_elo.</p><p><b>מה רואים:</b> לתיקו: 25.49% חזוי מול 26.85% בפועל, על אותם 3721 משחקים מחוץ למדגם.</p><p><b>הסבר נתונים:</b> שיעור חזוי הוא ממוצע ההסתברויות, לא ספירת argmax. Poisson בלתי תלוי מניח, בהינתן λ, שאין תלות בין שערי הקבוצות. המודל יכול לתאר היטב ממוצעי שערים ובכל זאת לפספס חלק ממבנה התיקו.</p><p><b>הסבר כדורגלני:</b> משחק כדורגל מגיב למצב שלו: התנהגות הקבוצות כשהן שוות או בפיגור עשויה לשנות את קצב ההבקעה. ההסבר מתאים לממצא, אך אינו נבדק כאן סיבתית.</p></div>

In [ ]:
draw_rates, draw_strata, draw_scores, draw_totals = draw_diagnostics(form_review.predictions)
display(form_table(draw_rates, percent=("predicted", "observed")))

<div dir="rtl"><h2>B2.6א — באילו סוגי משחקים חסר תיקו</h2><p><b>מה התא עושה:</b> מחלק לשלישים לפי הערך המוחלט של ההפרש בין λ הבית ל־λ החוץ.</p><p><b>מה רואים:</b> שלוש קבוצות דומות בגודל: שקול, בינוני ולא שקול. בבינוני כ־26.8% תיקו חזוי מול כ־30.2% שנצפה.</p><p><b>הסבר נתונים:</b> החלוקה נעשית רק לפי ציפיות השערים לפני המשחק, ולא לפי התוצאה שהתגלתה אחריו. בטבלה מופיעים גבולות ההפרש ומספר המשחקים. שמות השלישים יחסיים למדגם, אינם ספי כדורגל שנקבעו למודל.</p><p><b>הסבר כדורגלני:</b> החסר אינו אחיד: במשחקים עם פער בינוני בכוח השערים מתקבלים יותר תיקו מכפי שההנחה הבלתי תלויה נותנת.</p></div>

In [ ]:
display(form_table(draw_strata, percent=("predicted", "observed")))

<div dir="rtl"><h2>B2.6ב — תיקו מדויק וממוצע שערים: בדיקות שונות</h2><p><b>מה התא עושה:</b> גוזר את הסתברות 0:0, 1:1, 2:2 ו־3:3 מה־λ השמור ומשווה לתוצאות בפועל.</p><p><b>מה רואים:</b> יחס גדול מ־1 פירושו שנצפה יותר מהצפוי. לצד זה: כ־2.47 שערים חזויים למשחק מול כ־2.49 בפועל.</p><p><b>הסבר נתונים:</b> התוצאות המדויקות משתמשות באותה רשת מנורמלת <bdi dir="ltr">0–10</bdi> ששימשה ל־H/D/A בניסוי. ממוצע סך השערים הוא λ הבית ועוד λ החוץ. התאמת הממוצע אינה מבטיחה התאמה של כל תוצאה מדויקת.</p><p><b>הסבר כדורגלני:</b> זה הגשר ל־M3 לפי ה־ס13: לבדוק תיקון לתלות בתוצאות נמוכות, למשל Dixon-Coles, או ניפוח תיקו. לא מוסיפים תיקון כזה עכשיו, כדי לשמור על הניסוי שנשפט.</p></div>

In [ ]:
display(form_table(draw_scores, percent=("predicted", "observed")))
display(form_table(draw_totals))

<div dir="rtl"><h2>B2.7 — סיכום פעימה B וגשר ל־C</h2><p><b>מה התא עושה:</b> מרכז את ההחלטות שכבר אושרו ואת שני סרגלי ההמשך.</p><p><b>מה רואים:</b> E0050, התקפה והגנה ewma5/venue, וכושר בנקודות רק כמועמד XGBoost. סרגל התוצאות נשאר 1.0204; סרגל השערים הוא NLL ‏1.4095.</p><p><b>הסבר נתונים:</b> הבחירות התקבלו בפיתוח וב־QA; המחברת מציגה אותן ואינה בוחרת מחדש. פעימה C תוסיף הקשר t−1 — שווי, מיקום ספורטיבי, נקודות והפרש — השלמות לעולות, שכבת פיצ׳רים מאוחדת ופקודת ligat run prematch. אלה תכנון המשך, לא קוד שממומש כאן.</p><p><b>הסבר כדורגלני:</b> יש כעת זיכרון של כוח מתוצאות, ולצדו קצב הבקעה וספיגה עדכני לבית ולחוץ. ההקשר מהעונה הקודמת ישלים בהמשך את התמונה, עם מיקום ספורטיבי ונקודות על המגרש.</p></div>

In [ ]:
display(form_table(decision_table()))

<div dir="rtl"><h2>B2 — מספרי בקרה ושמירת גרפים</h2><p><b>מה התא עושה:</b> מציג את עוגני הבקרה ושומר את גרפי B2 בלבד כ־PNG ו־HTML.</p><p><b>מה רואים:</b> שני ייחוסי P, הנבחר ו־z, ייחוס O ו־0 מתוך 22, שיעורי התיקו ושלושת ערכי ρ של all לקראת 2023/2024.</p><p><b>הסבר נתונים:</b> כל המספרים מחושבים מהתוצרים שאומתו בתחילת B2. B5c הוא הסקירה המשותפת הבאה; העברת התחזיות ופעולות Git נשארות ל־B6.</p><p><b>הסבר כדורגלני:</b> אין משמעות כדורגלנית לפעולת השמירה; הדוח והתמונות מאפשרים לבדוק את ההסבר לפני ההמשך.</p></div>

In [ ]:
display(form_table(displayed_controls(form_review)))
b2_figure_dir = Path("reports/figures/prematch_b2")
b2_figure_dir.mkdir(parents=True, exist_ok=True)
for name, figure in b2_figures.items():
    figure.write_html(b2_figure_dir / f"{name}.html", include_plotlyjs=True)
    figure.write_image(
        b2_figure_dir / f"{name}.png", width=1800, height=figure.layout.height or 650, scale=1.5
    )
display(
    HTML(
        f'<div dir="rtl">נשמרו {len(b2_figures)} גרפים. זמן חלק B2: '
        f"{time.perf_counter() - b2_started:.2f} שניות.</div>"
    )
)